In [4]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

from langchain.chains import RetrievalQA
from langchain.llms import OpenAI
import os

import nltk

from langchain.document_loaders import DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import Chroma


# API 키 정보 로드
load_dotenv()


True

In [5]:
def add_to_chroma_batched(texts, embeddings, batch_size=32):
    db = None  # 초기화
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        if db is None:
            db = Chroma.from_documents(batch_texts, embeddings)
        else:
          db.add_documents(batch_texts)

    return db

In [6]:
nltk.download('averaged_perceptron_tagger_eng')

# 텍스트 파일이 있는 폴더 경로
folder_path = "./data/crawled_data"

# 텍스트 파일 로드
loader = DirectoryLoader(folder_path, glob="./*.txt")
documents = loader.load()

# 텍스트 분할
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
texts = text_splitter.split_documents(documents)[:]

# 임베딩 모델 및 벡터 데이터베이스 설정
embeddings = SentenceTransformerEmbeddings(model_name="paraphrase-multilingual-mpnet-base-v2")

chroma_db = add_to_chroma_batched(texts, embeddings, batch_size=64)

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\SSAFY\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [9]:
retriever = chroma_db.as_retriever(search_kwargs={'k': 3})

# 언어 모델 및 RAG 체인 설정
llm = OpenAI(max_tokens=256)
qa = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever)

# 질문 입력 및 답변 생성
query = """
당신은 아이들의 연령, 관심사, 상황에 맞춰 활동을 추천해주는 전문가입니다.
사용자는 당신이 추천한 활동을 아이들에게 시키고, 수행을 완료했다면 보상을 지급할 것입니다.
단순 재미보다는 아이들이 책임감을 가지고 발달에 도움이 되는 활동을 추천해주세요.

[출력 형식]:
1. 활동 제목: ...
2. 활동 설명: ...
3. 필요한 준비물 (선택 사항): ...
4. 예상 소요 시간 (선택 사항): ...
5. 추가 팁 (선택 사항): ..."""
result = qa.run(query)

print(result)

 
1. 활동 제목: "집안일 놀이로 하는 집콕육아"
2. 활동 설명: 이 활동은 아이들이 집안일을 함께 할 수 있도록 유도하고, 책임감을 가지게 하는 놀이입니다. 아이들이 자신의 연령에 맞게 집안일을 수행하고 완료했다면 보상을 받을 수 있습니다.
3. 필요한 준비물: 가구를 건드리거나 다치지 않도록 안전한 환경 조성, 간단한 집안일 도구 (빗자루, 걸레 등)
4. 예상 소요 시간: 30분 이상 (아이들의 연령과 수행하는 집안일의 종류에 따라 달라질 수 있음)
5. 추가 팁: 아이들이 수행할 집
